# Class 03 · Part 1: Getting gridded data

In class 02 a station gave you the weather at **one place**. Now you want a **map**: values everywhere,
e.g. the temperature of all of Europe. That is **gridded data**. In this notebook you learn

1. what a **grid** is and what **E-OBS** is,
2. **where** E-OBS is and what its files are called,
3. how to **download** a file with a function, and
4. what is inside a **netCDF** file, the file format of most climate data.

In part 2, `class03_use_gridded_data.ipynb`, you make maps and time series.

Run the cells from top to bottom with `Shift + Enter`.

## 1. A grid and E-OBS

A **lon-lat grid** cuts the map into cells along lines of longitude (west–east) and latitude (south–north),
like the squares of a chess board. Every cell has one value per day. E-OBS uses cells of **0.25°** (or 0.1°):

```
latitude
  75°N ┌──┬──┬──┬──┐
       ├──┼──┼──┼──┤      one cell: 0.25° × 0.25°,
       ├──┼──┼──┼──┤      one value per day
  25°N └──┴──┴──┴──┘
      40°W          75°E  longitude
```

**E-OBS** is made from the measurements of thousands of weather stations in Europe, like the stations of
class 02, collected by the project ECA&D and spread onto the grid. So it is still based on
observations, but now you have a value in every cell, also between the stations. E-OBS covers the land only:
over the sea there are no values.

## 2. Where is the data?

Open the download page of E-OBS in a new tab and look at it:

**https://surfobs.climate.copernicus.eu/dataaccess/access_eobs_chunks.php**

There is one file for each **variable**, **grid size** and **period of 15 years**. The variables have short names:

| Name | Variable | Unit |
|---|---|---|
| `tg` | daily mean temperature | °C |
| `tn`, `tx` | daily minimum and maximum temperature | °C |
| `rr` | daily precipitation | mm |
| `pp` | sea level pressure | hPa |
| `hu` | relative humidity | % |
| `fg` | wind speed | m/s |
| `qq` | global radiation | W/m² |

And the file name tells you what is inside:

```
tg_ens_mean_0.25deg_reg_2011-2025_v33.0e.nc
│  │        │           │         │      └ .nc: a netCDF file
│  │        │           │         └ version of E-OBS
│  │        │           └ period: 2011 to 2025
│  │        └ grid: cells of 0.25° on a regular lon-lat grid
│  └ ens_mean: the mean of an ensemble (several estimates; "spread" files tell how different they are)
└ tg: daily mean temperature
```

**Size matters:** a file of 0.25° is about **165 MB**, the same at 0.1° about **900 MB**. We use 0.25°.

**Terms of use:** E-OBS is free for non-commercial research and education, like this workshop. Whenever you
publish something made with it, acknowledge E-OBS and cite Cornes et al. (2018); the exact text is on the
download page and on the class page.

## 3. Download with a function

As for the stations in class 02, the course has a function that downloads a file for you:
`download_eobs`. Read its description:

In [ ]:
from utils import download_eobs

help(download_eobs)

And its code. You know the steps from class 02: look for the file on the web page, download it, save it.
Two things are new:

- The **version** is part of the file name and changes when E-OBS is updated, so the function looks for the
  file among the links of the download page, and takes the newest period.
- The file is large, so it is downloaded **piece by piece** (`iter_content`), and saved under a temporary
  name (`.part`) until it is complete. If your internet stops in the middle, the broken file is never taken
  for a good one: the next time, the function simply downloads it again.

In [ ]:
import inspect

print(inspect.getsource(download_eobs))

Now download the daily mean temperature, `"tg"`. The first time it takes a few minutes (165 MB); after that,
the function finds the file in `data` and returns at once.

In [ ]:
tg_file = download_eobs("tg")
tg_file

## 4. What is inside a netCDF file?

**netCDF** (`.nc`) is the usual file format of climate data. It holds the data *and* its description:
names, units, the grid. The package `xarray` (short `xr`) opens it.

`chunks={"time": 31}` tells xarray to read the file **31 days at a time**, only when it needs the values.
The whole file in memory would take 2 GB; this way it never does.

In [ ]:
import xarray as xr

ds = xr.open_dataset(tg_file, chunks={"time": 31})
ds

Click on the small triangles and icons above to look inside. A netCDF file has:

- **Dimensions**: the axes of the data: `time` (days), `latitude` and `longitude`.
- **Coordinates**: the values along the axes: the dates, and the latitude and longitude of every cell.
- **Data variables**: the data itself, here `tg`, a 3D array: one map per day.
- **Attributes**: the description, e.g. the units.

You get one variable with `ds["tg"]`, and its description with `.attrs`:

In [ ]:
ds["tg"].attrs

In [ ]:
print("days:", ds["time"].size)
print("from", ds["time"].values[0], "to", ds["time"].values[-1])
print("cells:", ds["latitude"].size, "latitudes ×", ds["longitude"].size, "longitudes")

## 5. A first map

`.sel(time="2025-07-15")` selects one day. `.plot()` draws it: xarray knows the latitudes and longitudes, so
the plot is already a map. The white areas are the sea (no values).

In [ ]:
ds["tg"].sel(time="2025-07-15").plot()

## 6. Exercise

1. Download the daily **precipitation**, `"rr"`, with `download_eobs`.
2. Open it with `xr.open_dataset` (with `chunks={"time": 31}`) and look inside: what is the name of its
   variable, and its unit?
3. Plot the precipitation of one day you choose. Why does it look so different from the temperature?

In [ ]:
# 1. download the precipitation

In [ ]:
# 2. open it and look inside

In [ ]:
# 3. a map of one day

## Next

Open **`class03_use_gridded_data.ipynb`**: maps of a month and of a year, and the temperature of Europe as
a time series.